# Cahn--Hilliard 2D: AWFNO-H10-K10-R2

This notebook supplies the first **10 exact states** once, predicts the next **10 states per AWFNO call**, and recursively updates the 10-state history using predictions. It uses the same split and absolute physical $L^2$ metrics as the existing benchmark.

Training uses exact H10-to-K10 windows, which is the direct H10 extension of the original AWFNO local-window objective.


In [9]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuration

The defaults run the complete $N=25,50,100,200,400$ sweep. For a quick test, set `RUN_SAMPLE_SIZES` to `25`. Completed and intermediate checkpoints are reused automatically.


In [10]:
import os
os.environ.setdefault('HISTORY_LENGTH', '10')
os.environ.setdefault('WINDOW_K', '10')
os.environ.setdefault('WINDOW_STRIDE', '10')
os.environ.setdefault('LOCAL_RADIUS', '2')
os.environ.setdefault('RUN_SAMPLE_SIZES', '25,50,100,200,400')
os.environ.setdefault('EPOCHS', '200')
os.environ.setdefault('BATCH_SIZE', '32')
os.environ.setdefault('FORCE_RETRAIN', '0')
os.environ.setdefault('SAVE_PLOTS', '1')
os.environ.setdefault('SAVE_ROLLOUT_ARRAYS', '0')


'0'

## Run AWFNO-H10-K10-R2

Fourier operations remain in float32/complex64; mixed precision is intentionally not used because CUDA does not support all complex-half operations.


In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Cahn--Hilliard 2D AWFNO-H10 benchmark.

This script evaluates one predictor:
    AWFNO-H10-K10-R2: ten input states and ten predicted states per recursive call

Expected dataset keys:
    X_input  : array of shape (N, Nx, Ny, 1)
    Y_output : array of shape (N, T, Nx, Ny, 1)

Internal layout:
    U has shape (N, Nx, Ny, Nt), with Nt = T + 1 and U[..., 0] = X_input.

Evaluation protocol:
    - train, validation, and test trajectories are disjoint;
    - the same test set is used for all sample sizes;
    - the temporal basis is computed from training windows only;
    - test trajectories are evaluated without test-window fitting or basis projection;
    - the first ten states are supplied exactly at inference;
    - all later states are recursive AWFNO predictions;
    - each call predicts a K=10 future window.

Training objective:
    AWFNO uses its original local K-window loss with a final-window term.

Primary metric:
    absolute physical L2 error with trapezoidal time quadrature.
"""

import os, gc, json, copy, time, random, hashlib
from pathlib import Path
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def env_bool(name, default=False):
    value = os.environ.get(name, None)
    if value is None:
        return bool(default)
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}
def env_str(name, default):
    return os.environ.get(name, default)


def env_int(name, default):
    value = os.environ.get(name, None)
    return int(value) if value not in (None, "") else int(default)


def env_float(name, default):
    value = os.environ.get(name, None)
    return float(value) if value not in (None, "") else float(default)


def env_list_int(name, default_list):
    value = os.environ.get(name, None)
    if value is None or str(value).strip() == "":
        return list(default_list)
    return [int(x.strip()) for x in str(value).split(",") if x.strip()]

def env_int_list(name, default):
    raw = os.environ.get(name, None)
    if raw is None:
        return list(default)
    raw = raw.strip()
    if not raw:
        return list(default)
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


# ============================================================
# 1. CONFIG
# ============================================================

SEED = env_int("SEED", 1234)


DT_PHYSICAL = env_float("DT_PHYSICAL", 0.01)
X_LEFT, X_RIGHT = 0.0, 1.0
Y_LEFT, Y_RIGHT = 0.0, 1.0

SAMPLE_SIZES = env_list_int("SAMPLE_SIZES", [25, 50, 100, 200, 400])
N_TRAIN_POOL = max(SAMPLE_SIZES)
N_VAL = env_int("N_VAL", 30)
N_TEST = env_int("N_TEST", 70)
SPLIT_SEED = env_int("SPLIT_SEED", 2026)
STRICT_REQUIRE_ENOUGH_SAMPLES = env_bool("STRICT_REQUIRE_ENOUGH_SAMPLES", True)

HISTORY_LENGTH = env_int("HISTORY_LENGTH", 10)
WINDOW_K = env_int("WINDOW_K", 10)
WINDOW_STRIDE = env_int("WINDOW_STRIDE", WINDOW_K)

BATCH_SIZE = env_int("BATCH_SIZE", 32)
EPOCHS = env_int("EPOCHS", 200)
LR = env_float("LR", 1e-3)
WEIGHT_DECAY = env_float("WEIGHT_DECAY", 1e-6)

WIDTH = env_int("WIDTH", 32)
N_LAYERS = env_int("N_LAYERS", 4)
MODES_X = env_int("MODES_X", 16)
MODES_Y = env_int("MODES_Y", 16)
LOCAL_RADIUS = env_int("LOCAL_RADIUS", 5)

LOSS_KIND = env_str("LOSS_KIND", "mse")
HUBER_BETA = env_float("HUBER_BETA", 0.02)
FINAL_LOSS_WEIGHT = env_float("FINAL_LOSS_WEIGHT", 1.0)
GRAD_LOSS_WEIGHT = env_float("GRAD_LOSS_WEIGHT", 0.0)
NORMALIZE_LOSS = env_bool("NORMALIZE_LOSS", True)

# For Cahn-Hilliard, mean conservation is useful for windowed rollout.
ENFORCE_MEAN = env_bool("ENFORCE_MEAN", True)

FORCE_RETRAIN = env_bool("FORCE_RETRAIN", False)
RUN_SAMPLE_SIZES = env_list_int("RUN_SAMPLE_SIZES", SAMPLE_SIZES)
CHECKPOINT_EVERY = env_int("CHECKPOINT_EVERY", 10)
INFERENCE_REPEATS = env_int("INFERENCE_REPEATS", 3)
USE_SPLIT_CACHE = env_bool("USE_SPLIT_CACHE", True)
SAVE_SPLIT_CACHE = env_bool("SAVE_SPLIT_CACHE", True)

RUN_EXACT_HASH_OVERLAP_CHECK = env_bool("RUN_EXACT_HASH_OVERLAP_CHECK", True)
RUN_DUPLICATE_REPORT_INSIDE_SPLITS = env_bool("RUN_DUPLICATE_REPORT_INSIDE_SPLITS", True)
RUN_ROLLOUT_INPUT_GAP_CHECK = env_bool("RUN_ROLLOUT_INPUT_GAP_CHECK", True)
DEBUG_SHUFFLE_TRAIN_OUTPUTS = env_bool("DEBUG_SHUFFLE_TRAIN_OUTPUTS", False)
DEBUG_RANDOM_BASIS = env_bool("DEBUG_RANDOM_BASIS", False)
INPUT_GAP_TOL = env_float("INPUT_GAP_TOL", 1e-12)

SAVE_PLOTS = env_bool("SAVE_PLOTS", True)
MAKE_TRAINING_CURVE_PLOTS = env_bool("MAKE_TRAINING_CURVE_PLOTS", True)
MAKE_L2_PER_TIME_PLOT = env_bool("MAKE_L2_PER_TIME_PLOT", True)
MAKE_FIELD_PLOTS = env_bool("MAKE_FIELD_PLOTS", True)
SHOW_PLOTS = env_bool("SHOW_PLOTS", False)
PLOT_TIME_INDICES_MANUAL_RAW = env_str("PLOT_TIME_INDICES", "5,10,final")
MAX_TEST_CASES_TO_PLOT = env_int("MAX_TEST_CASES_TO_PLOT", 8)
PLOT_DPI = env_int("PLOT_DPI", 140)
L2_CHUNK_SIZE = env_int("L2_CHUNK_SIZE", 16)

# False by default to keep memory and output usage controlled.
SAVE_ROLLOUT_ARRAYS = env_bool("SAVE_ROLLOUT_ARRAYS", False)

PLOT_TIME_INDICES_MANUAL = []
for item in PLOT_TIME_INDICES_MANUAL_RAW.split(","):
    item = item.strip()
    if not item:
        continue
    if item.lower() == "final":
        PLOT_TIME_INDICES_MANUAL.append("final")
    else:
        PLOT_TIME_INDICES_MANUAL.append(int(item))

DATA_FILE = Path("/content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt")
OUTPUT_ROOT = Path("/content/drive/MyDrive/anus/cahn_hilliard_awfno_h10_k10_r2_v1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DATA_FILE =", DATA_FILE)
print("exists    =", DATA_FILE.exists())
print("OUTPUT    =", OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DATA_FILE =", DATA_FILE)
print("exists    =", DATA_FILE.exists())
print("OUTPUT    =", OUTPUT_ROOT)

CACHE_ROOT = OUTPUT_ROOT / (
    f"AWFNO_H{HISTORY_LENGTH}_K{WINDOW_K}_W{WIDTH}_L{N_LAYERS}_"
    f"mx{MODES_X}_my{MODES_Y}_R{LOCAL_RADIUS}_E{EPOCHS}_"
    f"sizes{'-'.join(map(str, SAMPLE_SIZES))}_mean{int(ENFORCE_MEAN)}"
)

MODEL_DIR = CACHE_ROOT / "models"
PLOT_DIR = CACHE_ROOT / "plots"
DATA_DIR = CACHE_ROOT / "data"
RESULTS_DIR = CACHE_ROOT / "results"
for d in [MODEL_DIR, PLOT_DIR, DATA_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Matplotlib style.
plt.rcdefaults()
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "legend.fontsize": 10,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 110,
    "lines.linewidth": 2.0,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


DATA_FILE = /content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
exists    = True
OUTPUT    = /content/drive/MyDrive/anus/cahn_hilliard_awfno_h10_k10_r2_v1
DATA_FILE = /content/drive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
exists    = True
OUTPUT    = /content/drive/MyDrive/anus/cahn_hilliard_awfno_h10_k10_r2_v1


In [12]:
# ============================================================
# 2. REPRODUCIBILITY / DEVICE
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# 3. GLOBAL GEOMETRY
# ============================================================

x_grid_np = None
y_grid_np = None
t_grid = None
Nx = Ny = Nt = None
DX = DY = DT_SNAPSHOT = None
BASE_TAG = "uninitialized"


def set_geometry_from_arrays(U_shape, x_grid, y_grid, dt_physical):
    global x_grid_np, y_grid_np, t_grid, Nx, Ny, Nt, DX, DY, DT_SNAPSHOT
    if len(U_shape) != 4:
        raise ValueError(f"Expected U shape (N,Nx,Ny,Nt). Got {U_shape}")

    _, Nx, Ny, Nt = U_shape
    x_grid_np = np.asarray(x_grid, dtype=np.float32).reshape(-1)
    y_grid_np = np.asarray(y_grid, dtype=np.float32).reshape(-1)

    if len(x_grid_np) != Nx:
        raise ValueError(f"x_grid has len {len(x_grid_np)}, expected Nx={Nx}")
    if len(y_grid_np) != Ny:
        raise ValueError(f"y_grid has len {len(y_grid_np)}, expected Ny={Ny}")

    DX = float(np.mean(np.diff(x_grid_np.astype(np.float64)))) if Nx > 1 else 1.0
    DY = float(np.mean(np.diff(y_grid_np.astype(np.float64)))) if Ny > 1 else 1.0
    DT_SNAPSHOT = float(dt_physical)
    t_grid = np.arange(Nt, dtype=np.float32) * DT_SNAPSHOT

    print("\nGeometry")
    print("--------")
    print("N, Nx, Ny, Nt:", U_shape[0], Nx, Ny, Nt)
    print("x range      :", float(x_grid_np.min()), float(x_grid_np.max()))
    print("y range      :", float(y_grid_np.min()), float(y_grid_np.max()))
    print("t range      :", float(t_grid.min()), float(t_grid.max()))
    print("DX           :", DX)
    print("DY           :", DY)
    print("DT           :", DT_SNAPSHOT)


# ============================================================
# 4. BASIC HELPERS
# ============================================================

def short_hash(obj):
    txt = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.md5(txt.encode("utf-8")).hexdigest()[:10]


def clean_state_dict(sd):
    sd = copy.deepcopy(sd)
    if isinstance(sd, dict) and "_metadata" in sd:
        del sd["_metadata"]
    if hasattr(sd, "_metadata"):
        try:
            del sd._metadata
        except Exception:
            pass
    return OrderedDict((k, v) for k, v in sd.items() if k != "_metadata")


def torch_load_safe(path, map_location=None):
    # PyTorch versions differ in the weights_only argument.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def save_model_checkpoint(path, model, meta=None):
    torch.save({"state_dict": clean_state_dict(model.state_dict()), "meta": meta or {}}, path)


def load_model_checkpoint(path, model):
    payload = torch_load_safe(path, map_location=device)
    model.load_state_dict(clean_state_dict(payload["state_dict"]), strict=True)
    return payload.get("meta", {})


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def gb(nbytes):
    return nbytes / 1024**3


def print_array_info(name, arr):
    print(
        f"{name:16s}: shape={arr.shape}, dtype={arr.dtype}, "
        f"memory={gb(arr.nbytes):.4f} GB, "
        f"min={float(np.nanmin(arr)):+.6e}, "
        f"max={float(np.nanmax(arr)):+.6e}, "
        f"mean={float(np.nanmean(arr)):+.6e}, "
        f"std={float(np.nanstd(arr)):.6e}"
    )


def time_quadrature_weights(nt):
    w = np.ones(nt, dtype=np.float64)
    if nt > 1:
        w[0] = 0.5
        w[-1] = 0.5
    return w


# ============================================================
# 5. LOAD DATA / SPLIT / HASH CHECKS
# ============================================================

def load_cahn_hilliard_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset file not found:\n{path}\n\n"
            "Set DATA_FILE before running, for example:\n"
            "os.environ['DATA_FILE'] = '/path/to/cahn_hilliard_train_grid_16x16.pt'"
        )

    data = torch_load_safe(path, map_location="cpu")

    if "X_input" not in data or "Y_output" not in data:
        raise KeyError("Expected keys 'X_input' and 'Y_output'.")

    X = data["X_input"].float()       # (N,Nx,Ny,1)
    Y = data["Y_output"].float()      # (N,T,Nx,Ny,1)

    if X.ndim != 4 or X.shape[-1] != 1:
        raise ValueError(f"Expected X_input (N,Nx,Ny,1), got {tuple(X.shape)}")
    if Y.ndim != 5 or Y.shape[-1] != 1:
        raise ValueError(f"Expected Y_output (N,T,Nx,Ny,1), got {tuple(Y.shape)}")

    if "x_grid" in data:
        x_grid = data["x_grid"].float().reshape(-1).numpy()
    elif "xg" in data:
        x_grid = data["xg"].float().reshape(-1).numpy()
    else:
        x_grid = np.linspace(X_LEFT, X_RIGHT, X.shape[1], endpoint=False, dtype=np.float32)

    if "y_grid" in data:
        y_grid = data["y_grid"].float().reshape(-1).numpy()
    elif "yg" in data:
        y_grid = data["yg"].float().reshape(-1).numpy()
    else:
        y_grid = np.linspace(Y_LEFT, Y_RIGHT, X.shape[2], endpoint=False, dtype=np.float32)

    X_np = X[..., 0].contiguous().numpy().astype(np.float32)  # (N,Nx,Ny)
    Y_np = Y[..., 0].permute(0, 2, 3, 1).contiguous().numpy().astype(np.float32)  # (N,Nx,Ny,T)
    U = np.concatenate([X_np[..., None], Y_np], axis=-1).astype(np.float32)       # (N,Nx,Ny,T+1)

    if not np.all(np.isfinite(U)):
        raise ValueError("Loaded U contains NaN or Inf.")

    sample_ids = data.get("sample_ids", None)
    if sample_ids is None:
        sample_ids = np.arange(U.shape[0], dtype=np.int64)
    elif torch.is_tensor(sample_ids):
        sample_ids = sample_ids.long().numpy()
    else:
        sample_ids = np.asarray(sample_ids)

    print("\nLoaded Cahn-Hilliard data")
    print("-------------------------")
    print("path    :", path)
    print("X_input :", tuple(X.shape), "->", X_np.shape)
    print("Y_output:", tuple(Y.shape), "->", Y_np.shape)
    print("U       :", U.shape, "= (N,Nx,Ny,Nt)")
    print("sample_ids:", sample_ids.shape)

    return U, x_grid.astype(np.float32), y_grid.astype(np.float32), sample_ids


def split_cache_path():
    data_sig = {
        "DATA_FILE_name": Path(DATA_FILE).name,
        "DATA_FILE_size": Path(DATA_FILE).stat().st_size if Path(DATA_FILE).exists() else None,
        "SAMPLE_SIZES": SAMPLE_SIZES,
        "N_TRAIN_POOL": N_TRAIN_POOL,
        "N_VAL": N_VAL,
        "N_TEST": N_TEST,
        "SPLIT_SEED": SPLIT_SEED,
    }
    tag = short_hash(data_sig)
    return DATA_DIR / f"split_CH2D_{tag}.npz"


def make_split_indices(n_total):
    need = N_TRAIN_POOL + N_VAL + N_TEST
    if need > n_total:
        msg = (
            f"Need {need} trajectories (N_TRAIN_POOL={N_TRAIN_POOL}, "
            f"N_VAL={N_VAL}, N_TEST={N_TEST}), but dataset has only {n_total}. "
            "Reduce SAMPLE_SIZES, N_VAL, or N_TEST."
        )
        if STRICT_REQUIRE_ENOUGH_SAMPLES:
            raise ValueError(msg)
        print("WARNING:", msg)

    rng = np.random.default_rng(SPLIT_SEED)
    perm = rng.permutation(n_total)
    selected = perm[:min(need, n_total)]

    train_idx = selected[:N_TRAIN_POOL]
    val_idx = selected[N_TRAIN_POOL:N_TRAIN_POOL + N_VAL]
    test_idx = selected[N_TRAIN_POOL + N_VAL:N_TRAIN_POOL + N_VAL + N_TEST]

    return train_idx, val_idx, test_idx


def load_split_cache():
    path = split_cache_path()
    if not USE_SPLIT_CACHE or not path.exists():
        return None

    print("\nLoading split cache")
    print("-------------------")
    print(path)

    z = np.load(path, allow_pickle=True)
    return (
        z["U_train_pool"].astype(np.float32),
        z["U_val"].astype(np.float32),
        z["U_test"].astype(np.float32),
        z["train_idx"],
        z["val_idx"],
        z["test_idx"],
        z["x_grid"].astype(np.float32),
        z["y_grid"].astype(np.float32),
    )


def save_split_cache(U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid):
    if not SAVE_SPLIT_CACHE:
        return
    path = split_cache_path()
    print("\nSaving split cache")
    print("------------------")
    print(path)
    np.savez(
        path,
        U_train_pool=U_train_pool,
        U_val=U_val,
        U_test=U_test,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        x_grid=x_grid,
        y_grid=y_grid,
    )
    print("Saved:", path)


def load_or_create_split():
    cached = load_split_cache()
    if cached is not None:
        return cached

    U, x_grid, y_grid, sample_ids = load_cahn_hilliard_file(DATA_FILE)
    train_idx, val_idx, test_idx = make_split_indices(U.shape[0])

    U_train_pool = U[train_idx]
    U_val = U[val_idx]
    U_test = U[test_idx]

    print("\nSplit")
    print("-----")
    print("N total     :", U.shape[0])
    print("N train pool:", len(train_idx))
    print("N val       :", len(val_idx))
    print("N test      :", len(test_idx))
    print("train_idx first/last:", train_idx[:5], train_idx[-5:] if len(train_idx) else [])
    print("val_idx   first/last:", val_idx[:5], val_idx[-5:] if len(val_idx) else [])
    print("test_idx  first/last:", test_idx[:5], test_idx[-5:] if len(test_idx) else [])

    save_split_cache(U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid)

    return U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid


def trajectory_hashes(U):
    U32 = np.ascontiguousarray(U.astype(np.float32, copy=False))
    return [hashlib.sha1(U32[i].tobytes()).hexdigest() for i in range(U32.shape[0])]


def check_exact_overlap(Ua, Ub, name_a, name_b):
    ha = set(trajectory_hashes(Ua))
    hb = set(trajectory_hashes(Ub))
    inter = ha.intersection(hb)
    print(f"\nExact overlap check {name_a} vs {name_b}")
    print("------------------------------------------")
    print("duplicates:", len(inter))
    if inter:
        raise RuntimeError(f"Duplicate trajectories found between {name_a} and {name_b}.")


def report_duplicate_trajectories(U, name):
    hashes = trajectory_hashes(U)
    seen, duplicates = {}, []
    for i, h in enumerate(hashes):
        if h in seen:
            duplicates.append((seen[h], i))
        else:
            seen[h] = i
    print(f"\nDuplicate check inside {name}")
    print("--------------------------------")
    print("total samples :", len(hashes))
    print("unique samples:", len(seen))
    print("duplicates    :", len(duplicates))
    if duplicates:
        print("first duplicate pairs:", duplicates[:10])
    return duplicates


# ============================================================
# 6. L2 METRICS
# ============================================================

def abs_l2_window_batch(pred, true):
    # pred,true: (B,Nx,Ny,K)
    return torch.sqrt(torch.sum((pred - true) ** 2, dim=(1, 2, 3)) * DX * DY * DT_SNAPSHOT + 1e-30)


def abs_l2_field_batch(pred, true):
    # pred,true: (B,Nx,Ny)
    return torch.sqrt(torch.sum((pred - true) ** 2, dim=(1, 2)) * DX * DY + 1e-30)


def l2_per_timestep_np(pred, true, chunk_size=L2_CHUNK_SIZE):
    if pred.shape != true.shape:
        raise ValueError(f"Shape mismatch: pred {pred.shape}, true {true.shape}")
    n, nx, ny, nt = pred.shape
    out = np.zeros((n, nt), dtype=np.float64)
    for i0 in range(0, n, chunk_size):
        i1 = min(i0 + chunk_size, n)
        err = pred[i0:i1].astype(np.float32, copy=False) - true[i0:i1].astype(np.float32, copy=False)
        out[i0:i1] = np.sqrt(np.sum(err * err, axis=(1, 2), dtype=np.float64) * DX * DY + 1e-30)
        del err
        gc.collect()
    return out


def abs_l2_full_traj_np(pred, true):
    l2_t = l2_per_timestep_np(pred, true)
    w = time_quadrature_weights(l2_t.shape[1]).reshape(1, -1)
    return np.sqrt(np.sum(w * l2_t ** 2, axis=1) * DT_SNAPSHOT + 1e-30)


def abs_l2_final_np(pred, true):
    return l2_per_timestep_np(pred, true)[:, -1]


def rollout_l2_metrics(pred, true):
    full = abs_l2_full_traj_np(pred, true)
    final = abs_l2_final_np(pred, true)
    l2_t = l2_per_timestep_np(pred, true)
    return {
        "full_l2_mean": float(np.mean(full)),
        "full_l2_std": float(np.std(full)),
        "full_l2_median": float(np.median(full)),
        "final_l2_mean": float(np.mean(final)),
        "final_l2_std": float(np.std(final)),
        "final_l2_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "all_full_l2": full,
        "all_final_l2": final,
        "all_l2_t": l2_t,
    }


def mass_per_timestep_np(U):
    return np.mean(U, axis=(1, 2))


Using device: cuda


In [13]:
# ============================================================
# 7. H10 -> K10 WINDOWS
# ============================================================

def make_history_window_starts(nt, history_length, window_k, stride):
    last_start = nt - history_length - window_k
    if last_start < 0:
        raise ValueError(
            f"Need Nt >= H+K, got Nt={nt}, H={history_length}, K={window_k}."
        )
    return list(range(0, last_start + 1, stride))


def build_history_window_tensors(U, starts):
    n, nx, ny, nt = U.shape
    total = n * len(starts)
    history = np.empty(
        (total, nx, ny, HISTORY_LENGTH), dtype=np.float32
    )
    future = np.empty((total, nx, ny, WINDOW_K), dtype=np.float32)
    q = 0
    for i in range(n):
        for s in starts:
            history[q] = U[i, :, :, s:s + HISTORY_LENGTH]
            future[q] = U[
                i, :, :,
                s + HISTORY_LENGTH:s + HISTORY_LENGTH + WINDOW_K
            ]
            q += 1
    return history, future


class HistoryWindowDataset2d(Dataset):
    def __init__(self, history, future):
        self.history = torch.as_tensor(history, dtype=torch.float32)
        self.future = torch.as_tensor(future, dtype=torch.float32)

    def __len__(self):
        return self.history.shape[0]

    def __getitem__(self, idx):
        return {"history": self.history[idx], "future": self.future[idx]}


class GaussianNormalizer:
    def __init__(self, x=None, mean=None, std=None, eps=1.0e-6):
        if x is not None:
            self.mean = x.mean().detach().clone()
            self.std = x.std().detach().clone()
        else:
            self.mean = torch.tensor(float(mean), dtype=torch.float32)
            self.std = torch.tensor(float(std), dtype=torch.float32)
        self.eps = float(eps)

    def encode(self, x):
        return (x - self.mean) / (self.std + self.eps)

    def decode(self, x):
        return x * (self.std + self.eps) + self.mean

    def to(self, device_):
        self.mean = self.mean.to(device_)
        self.std = self.std.to(device_)
        return self

    def state(self):
        return {
            "mean": float(self.mean.detach().cpu()),
            "std": float(self.std.detach().cpu()),
            "eps": self.eps,
        }


# ============================================================
# 8. EXACT AWFNO ARCHITECTURE, WITH H INPUT CHANNELS
# ============================================================

class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes_x = modes_x
        self.modes_y = modes_y
        scale = 1.0 / np.sqrt(in_channels * out_channels)
        self.weights_pos = nn.Parameter(
            scale * torch.randn(
                in_channels, out_channels, modes_x, modes_y,
                dtype=torch.cfloat,
            )
        )
        self.weights_neg = nn.Parameter(
            scale * torch.randn(
                in_channels, out_channels, modes_x, modes_y,
                dtype=torch.cfloat,
            )
        )

    @staticmethod
    def compl_mul2d(a, b):
        return torch.einsum("bixy,ioxy->boxy", a, b)

    def forward(self, x):
        batch, _, nx, ny = x.shape
        x_ft = torch.fft.rfft2(x.float(), norm="ortho")
        out_ft = torch.zeros(
            batch, self.out_channels, nx, ny // 2 + 1,
            dtype=torch.cfloat, device=x.device,
        )
        mx = min(self.modes_x, nx)
        my = min(self.modes_y, ny // 2 + 1)
        if mx > 0 and my > 0:
            out_ft[:, :, :mx, :my] = self.compl_mul2d(
                x_ft[:, :, :mx, :my],
                self.weights_pos[:, :, :mx, :my],
            )
            out_ft[:, :, -mx:, :my] = self.compl_mul2d(
                x_ft[:, :, -mx:, :my],
                self.weights_neg[:, :, :mx, :my],
            )
        return torch.fft.irfft2(out_ft, s=(nx, ny), norm="ortho")


class LocalOperator2d(nn.Module):
    def __init__(self, width, radius):
        super().__init__()
        self.op = nn.Conv2d(
            width, width,
            kernel_size=2 * radius + 1,
            padding=radius,
            padding_mode="circular",
            bias=True,
        )

    def forward(self, z):
        return self.op(z)


class FourierOperator2d(nn.Module):
    def __init__(self, width, modes_x, modes_y):
        super().__init__()
        self.op = SpectralConv2d(width, width, modes_x, modes_y)

    def forward(self, z):
        return self.op(z)


class ExactLKLAlternationLayer2d(nn.Module):
    """Exact AFF ordering: local -> Fourier -> local, with residual updates."""

    def __init__(self, width, modes_x, modes_y, local_radius):
        super().__init__()
        self.L_l1 = LocalOperator2d(width, local_radius)
        self.K_l = FourierOperator2d(width, modes_x, modes_y)
        self.L_l2 = LocalOperator2d(width, local_radius)
        self.sigma = nn.GELU()

    def forward(self, v_l):
        v_l_1 = v_l + self.sigma(self.L_l1(v_l))
        v_l_2 = v_l_1 + self.sigma(torch.tanh(self.K_l(v_l_1)))
        return v_l_2 + self.sigma(self.L_l2(v_l_2))


def make_grid_buffer():
    x = torch.tensor(x_grid_np, dtype=torch.float32)
    y = torch.tensor(y_grid_np, dtype=torch.float32)
    xn = (x - x.min()) / (x.max() - x.min() + 1.0e-12)
    yn = (y - y.min()) / (y.max() - y.min() + 1.0e-12)
    X, Y = torch.meshgrid(xn, yn, indexing="ij")
    grid = torch.stack(
        [
            X, Y,
            torch.sin(2 * np.pi * X), torch.cos(2 * np.pi * X),
            torch.sin(2 * np.pi * Y), torch.cos(2 * np.pi * Y),
        ],
        dim=-1,
    )
    return grid.unsqueeze(0)


class AlternatingFNOHistoryEncoder2d(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("grid", make_grid_buffer(), persistent=False)
        self.input_proj = nn.Linear(HISTORY_LENGTH + 6, WIDTH)
        self.blocks = nn.ModuleList(
            [
                ExactLKLAlternationLayer2d(
                    WIDTH, MODES_X, MODES_Y, LOCAL_RADIUS
                )
                for _ in range(N_LAYERS)
            ]
        )

    def forward(self, history_norm):
        if history_norm.ndim != 4:
            raise ValueError(
                f"Expected history (B,Nx,Ny,H), got {history_norm.shape}."
            )
        batch = history_norm.shape[0]
        grid = self.grid.expand(batch, -1, -1, -1)
        z = torch.cat([history_norm, grid], dim=-1)
        z = self.input_proj(z).permute(0, 3, 1, 2).contiguous()
        for block in self.blocks:
            z = block(z)
        return z.permute(0, 2, 3, 1).contiguous()


class AWFNOHistory2d(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AlternatingFNOHistoryEncoder2d()
        self.fc1 = nn.Linear(WIDTH, 128)
        self.fc2 = nn.Linear(128, WINDOW_K)

    def forward(self, history_norm):
        z = self.encoder(history_norm)
        return self.fc2(F.gelu(self.fc1(z))).contiguous()


def count_real_scalars(model):
    total = 0
    for p in model.parameters():
        total += p.numel() * (2 if torch.is_complex(p) else 1)
    return int(total)


def enforce_mean_future(future, history):
    reference_mean = history[:, :, :, -1].mean(
        dim=(1, 2), keepdim=True
    ).unsqueeze(-1)
    return future + reference_mean - future.mean(
        dim=(1, 2), keepdim=True
    )


def make_predict(model, history_normalizer, future_normalizer):
    def predict(history):
        raw = model(history_normalizer.encode(history))
        future = future_normalizer.decode(raw)
        if ENFORCE_MEAN:
            future = enforce_mean_future(future, history)
        return future
    return predict


In [14]:
# ============================================================
# 9. LOSS AND TRAINING
# ============================================================

def data_loss_fn(pred, true):
    if LOSS_KIND == "mse":
        return F.mse_loss(pred, true)
    if LOSS_KIND == "huber":
        return F.smooth_l1_loss(pred, true, beta=HUBER_BETA)
    raise ValueError(f"Unknown LOSS_KIND={LOSS_KIND}")


def normalized_loss(pred, true, normalizer):
    if NORMALIZE_LOSS:
        return data_loss_fn(normalizer.encode(pred), normalizer.encode(true))
    return data_loss_fn(pred, true)


def window_loss(pred, true, future_normalizer):
    reconstruction = normalized_loss(pred, true, future_normalizer)
    final = normalized_loss(
        pred[:, :, :, -1], true[:, :, :, -1], future_normalizer
    )
    return reconstruction + FINAL_LOSS_WEIGHT * final


def abs_l2_window_batch(pred, true):
    return torch.sqrt(
        torch.sum((pred - true) ** 2, dim=(1, 2, 3))
        * DX * DY * DT_SNAPSHOT + 1.0e-30
    )


def abs_l2_field_batch(pred, true):
    return torch.sqrt(
        torch.sum((pred - true) ** 2, dim=(1, 2))
        * DX * DY + 1.0e-30
    )


def train_epoch(model, loader, optimizer, predict, future_normalizer):
    model.train()
    total_loss = 0.0
    total_l2 = 0.0
    n_seen = 0
    for batch in loader:
        history = batch["history"].to(device, non_blocking=True)
        true = batch["future"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = predict(history)
        loss = window_loss(pred, true, future_normalizer)
        if not torch.isfinite(loss):
            raise FloatingPointError("AWFNO-H10 produced a non-finite loss.")
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not torch.isfinite(grad_norm):
            raise FloatingPointError("AWFNO-H10 produced non-finite gradients.")
        optimizer.step()
        batch_size = history.shape[0]
        total_loss += float(loss.detach()) * batch_size
        total_l2 += float(abs_l2_window_batch(pred.detach(), true).mean()) * batch_size
        n_seen += batch_size
    return total_loss / n_seen, total_l2 / n_seen


@torch.no_grad()
def eval_epoch(model, loader, predict, future_normalizer):
    model.eval()
    total_loss = 0.0
    total_l2 = 0.0
    total_final = 0.0
    n_seen = 0
    for batch in loader:
        history = batch["history"].to(device, non_blocking=True)
        true = batch["future"].to(device, non_blocking=True)
        pred = predict(history)
        loss = window_loss(pred, true, future_normalizer)
        batch_size = history.shape[0]
        total_loss += float(loss) * batch_size
        total_l2 += float(abs_l2_window_batch(pred, true).mean()) * batch_size
        total_final += float(
            abs_l2_field_batch(pred[:, :, :, -1], true[:, :, :, -1]).mean()
        ) * batch_size
        n_seen += batch_size
    return (
        total_loss / n_seen,
        total_l2 / n_seen,
        total_final / n_seen,
    )


def byte_rng_state(value):
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        value = torch.from_numpy(value)
    elif not torch.is_tensor(value):
        value = torch.tensor(value)
    return value.detach().cpu().to(dtype=torch.uint8).contiguous()


def checkpoint_payload(
    model, optimizer, scheduler, epoch, history, best_val, best_epoch,
    elapsed, shuffle_generator, history_normalizer, future_normalizer,
):
    return {
        "state_dict": clean_state_dict(model.state_dict()),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": int(epoch),
        "history": history,
        "best_val": float(best_val),
        "best_epoch": int(best_epoch),
        "train_time_sec": float(elapsed),
        "shuffle_generator_state": shuffle_generator.get_state().cpu(),
        "history_normalizer": history_normalizer.state(),
        "future_normalizer": future_normalizer.state(),
        "config": {
            "history_length": HISTORY_LENGTH,
            "window_k": WINDOW_K,
            "width": WIDTH,
            "layers": N_LAYERS,
            "modes_x": MODES_X,
            "modes_y": MODES_Y,
            "local_radius": LOCAL_RADIUS,
            "epochs": EPOCHS,
        },
    }


def fit_or_load_awfno_h10(n_train, train_history, train_future,
                           val_history, val_future):
    run_dir = MODEL_DIR / f"N{n_train}"
    run_dir.mkdir(parents=True, exist_ok=True)
    latest_path = run_dir / "latest.pt"
    best_path = run_dir / "best.pt"
    done_path = run_dir / "done.pt"

    history_normalizer = GaussianNormalizer(
        torch.as_tensor(train_history, dtype=torch.float32)
    ).to(device)
    future_normalizer = GaussianNormalizer(
        torch.as_tensor(train_future, dtype=torch.float32)
    ).to(device)

    model = AWFNOHistory2d().to(device)
    predict = make_predict(model, history_normalizer, future_normalizer)

    if done_path.exists() and not FORCE_RETRAIN:
        saved = torch_load_safe(done_path, map_location=device)
        model.load_state_dict(clean_state_dict(saved["state_dict"]), strict=True)
        print(f"Loading completed checkpoint: {done_path}")
        return {
            "model": model,
            "predict": predict,
            "history": saved["history"],
            "best_val": float(saved["best_val"]),
            "best_epoch": int(saved["best_epoch"]),
            "train_time_sec": float(saved["train_time_sec"]),
            "checkpoint": str(done_path),
        }

    train_dataset = HistoryWindowDataset2d(train_history, train_future)
    val_dataset = HistoryWindowDataset2d(val_history, val_future)
    shuffle_generator = torch.Generator().manual_seed(SEED + n_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=shuffle_generator,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS
    )
    start_epoch = 1
    best_val = float("inf")
    best_epoch = 0
    elapsed_before = 0.0
    history = {
        "epoch": [],
        "train_loss": [],
        "train_l2": [],
        "val_loss": [],
        "val_l2": [],
        "val_final_l2": [],
    }

    if latest_path.exists() and not FORCE_RETRAIN:
        latest = torch_load_safe(latest_path, map_location=device)
        model.load_state_dict(clean_state_dict(latest["state_dict"]), strict=True)
        optimizer.load_state_dict(latest["optimizer"])
        scheduler.load_state_dict(latest["scheduler"])
        start_epoch = int(latest["epoch"]) + 1
        best_val = float(latest["best_val"])
        best_epoch = int(latest["best_epoch"])
        elapsed_before = float(latest.get("train_time_sec", 0.0))
        history = latest["history"]
        rng_state = byte_rng_state(latest.get("shuffle_generator_state"))
        if rng_state is not None:
            shuffle_generator.set_state(rng_state)
        print(f"Resuming N={n_train} at epoch {start_epoch}.")

    t0 = time.time()
    for epoch in range(start_epoch, EPOCHS + 1):
        train_loss, train_l2 = train_epoch(
            model, train_loader, optimizer, predict, future_normalizer
        )
        val_loss, val_l2, val_final = eval_epoch(
            model, val_loader, predict, future_normalizer
        )
        scheduler.step()

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_l2"].append(train_l2)
        history["val_loss"].append(val_loss)
        history["val_l2"].append(val_l2)
        history["val_final_l2"].append(val_final)

        elapsed = elapsed_before + time.time() - t0
        if val_l2 < best_val:
            best_val = val_l2
            best_epoch = epoch
            torch.save(
                {
                    "state_dict": clean_state_dict(model.state_dict()),
                    "best_val": best_val,
                    "best_epoch": best_epoch,
                },
                best_path,
            )

        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            print(
                f"[AWFNO-H{HISTORY_LENGTH}-K{WINDOW_K}] "
                f"N={n_train} {epoch:04d}/{EPOCHS} | "
                f"train loss {train_loss:.6e} | train L2 {train_l2:.6e} | "
                f"val loss {val_loss:.6e} | val L2 {val_l2:.6e} | "
                f"val final {val_final:.6e}"
            )

        if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
            torch.save(
                checkpoint_payload(
                    model, optimizer, scheduler, epoch, history,
                    best_val, best_epoch, elapsed, shuffle_generator,
                    history_normalizer, future_normalizer,
                ),
                latest_path,
            )

    if not best_path.exists():
        raise RuntimeError("No finite validation checkpoint was produced.")
    best = torch_load_safe(best_path, map_location=device)
    model.load_state_dict(clean_state_dict(best["state_dict"]), strict=True)
    train_time = elapsed_before + time.time() - t0
    done = {
        "state_dict": clean_state_dict(model.state_dict()),
        "history": history,
        "best_val": float(best_val),
        "best_epoch": int(best_epoch),
        "train_time_sec": float(train_time),
        "history_normalizer": history_normalizer.state(),
        "future_normalizer": future_normalizer.state(),
    }
    torch.save(done, done_path)
    return {
        "model": model,
        "predict": predict,
        "history": history,
        "best_val": float(best_val),
        "best_epoch": int(best_epoch),
        "train_time_sec": float(train_time),
        "checkpoint": str(done_path),
    }


In [15]:
# ============================================================
# 10. RECURSIVE H10/K10 ROLLOUT AND REPORTING
# ============================================================

@torch.no_grad()
def rollout_awfno_h10(model, predict, U_np, diagnostics=False):
    model.eval()
    n, nx, ny, nt = U_np.shape
    if nt <= HISTORY_LENGTH:
        raise ValueError("Trajectory does not contain forecast states.")
    output = np.zeros_like(U_np, dtype=np.float32)
    output[:, :, :, :HISTORY_LENGTH] = U_np[:, :, :, :HISTORY_LENGTH]

    for i0 in range(0, n, BATCH_SIZE):
        i1 = min(i0 + BATCH_SIZE, n)
        truth = U_np[i0:i1]
        history = torch.as_tensor(
            truth[:, :, :, :HISTORY_LENGTH],
            dtype=torch.float32,
            device=device,
        )
        cursor = HISTORY_LENGTH
        block = 0
        while cursor < nt:
            future = predict(history)
            use_k = min(WINDOW_K, nt - cursor)
            output[i0:i1, :, :, cursor:cursor + use_k] = (
                future[:, :, :, :use_k].detach().cpu().numpy()
            )
            if diagnostics:
                true_terminal = torch.as_tensor(
                    truth[:, :, :, cursor + use_k - 1],
                    dtype=torch.float32,
                    device=device,
                )
                terminal = abs_l2_field_batch(
                    future[:, :, :, use_k - 1], true_terminal
                ).mean()
                print(
                    f"batch {i0:03d}:{i1:03d}, block {block:02d}, "
                    f"times {cursor}:{cursor + use_k - 1}, "
                    f"terminal L2={float(terminal):.6e}"
                )

            if cursor + use_k < nt:
                new_values = future[:, :, :, :use_k].detach()
                if use_k >= HISTORY_LENGTH:
                    history = new_values[:, :, :, -HISTORY_LENGTH:]
                else:
                    history = torch.cat(
                        [history[:, :, :, use_k:], new_values], dim=-1
                    )
            cursor += use_k
            block += 1
    return output


def timed_test_rollout(model, predict, U_test):
    if device.type == "cuda":
        torch.cuda.synchronize()
    _ = rollout_awfno_h10(model, predict, U_test[:min(2, len(U_test))])
    times = []
    prediction = None
    for _ in range(INFERENCE_REPEATS):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        prediction = rollout_awfno_h10(model, predict, U_test)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return prediction, float(np.median(times)), times


def forecast_only_full_l2(pred, true):
    start = HISTORY_LENGTH - 1
    l2_t = l2_per_timestep_np(pred, true)[:, start:]
    weights = time_quadrature_weights(l2_t.shape[1])[None, :]
    values = np.sqrt(
        np.sum(weights * l2_t * l2_t, axis=1) * DT_SNAPSHOT + 1.0e-30
    )
    return {
        "forecast_full_mean": float(values.mean()),
        "forecast_full_std": float(values.std()),
        "forecast_full_median": float(np.median(values)),
    }


def plot_history(history, n_train):
    if not SAVE_PLOTS:
        return
    epochs = np.asarray(history["epoch"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].semilogy(epochs, history["train_loss"], label="train")
    axes[0].semilogy(epochs, history["val_loss"], label="validation")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("normalized H10-to-K10 window loss")
    axes[0].legend()
    axes[1].semilogy(epochs, history["train_l2"], label="train")
    axes[1].semilogy(epochs, history["val_l2"], label="validation")
    axes[1].semilogy(
        epochs, history["val_final_l2"], label="validation final-window"
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("absolute physical L2")
    axes[1].legend()
    fig.suptitle(f"AWFNO-H10-K10-R2 training, N={n_train}")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"N{n_train}_training.png", dpi=PLOT_DPI)
    plt.close(fig)


def plot_l2_time(metrics, n_train):
    if not SAVE_PLOTS:
        return
    fig, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.plot(t_grid, metrics["mean_t"], label="mean")
    axis.plot(t_grid, metrics["median_t"], "--", label="median")
    axis.fill_between(
        t_grid,
        np.maximum(metrics["mean_t"] - metrics["std_t"], 0.0),
        metrics["mean_t"] + metrics["std_t"],
        alpha=0.18,
        label="mean +/- std",
    )
    axis.axvline(
        t_grid[HISTORY_LENGTH - 1], color="black", alpha=0.5,
        linestyle=":", label="last exact input"
    )
    axis.set_xlabel("time")
    axis.set_ylabel("absolute physical L2")
    axis.set_title(f"AWFNO-H10-K10-R2 recursive rollout, N={n_train}")
    axis.legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"N{n_train}_l2_per_time.png", dpi=PLOT_DPI)
    plt.close(fig)


def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(type(value).__name__)


In [16]:
# ============================================================
# 11. DATA, SMOKE TEST, SAMPLE-SIZE SWEEP
# ============================================================

U_train_pool, U_val, U_test, train_idx, val_idx, test_idx, x_grid, y_grid = (
    load_or_create_split()
)
set_geometry_from_arrays(U_train_pool.shape, x_grid, y_grid, DT_PHYSICAL)

if RUN_EXACT_HASH_OVERLAP_CHECK:
    check_exact_overlap(U_train_pool, U_val, "train", "validation")
    check_exact_overlap(U_train_pool, U_test, "train", "test")
    check_exact_overlap(U_val, U_test, "validation", "test")

starts = make_history_window_starts(
    Nt, HISTORY_LENGTH, WINDOW_K, WINDOW_STRIDE
)
print("\nTemporal protocol")
print("-----------------")
print("exact input states :", HISTORY_LENGTH)
print("predicted per call :", WINDOW_K)
print("training starts    :", starts)
print("test forecast      : t[10:] recursively; t[:10] copied exactly")

val_history, val_future = build_history_window_tensors(U_val, starts)

smoke_model = AWFNOHistory2d().to(device)
with torch.no_grad():
    smoke_output = smoke_model(
        torch.zeros(
            2, Nx, Ny, HISTORY_LENGTH, dtype=torch.float32, device=device
        )
    )
assert smoke_output.shape == (2, Nx, Ny, WINDOW_K), smoke_output.shape
print("\nShape smoke test passed:", tuple(smoke_output.shape))
print("Trainable parameters     :", f"{count_params(smoke_model):,}")
print("Stored real scalars      :", f"{count_real_scalars(smoke_model):,}")
del smoke_model, smoke_output
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

all_results = []
for n_train in RUN_SAMPLE_SIZES:
    if n_train not in SAMPLE_SIZES:
        raise ValueError(f"Unexpected sample size {n_train}.")
    train_history, train_future = build_history_window_tensors(
        U_train_pool[:n_train], starts
    )
    print(
        f"\nN={n_train}: train histories {train_history.shape}, "
        f"targets {train_future.shape}"
    )
    fitted = fit_or_load_awfno_h10(
        n_train, train_history, train_future, val_history, val_future
    )
    prediction, inference_median, inference_all = timed_test_rollout(
        fitted["model"], fitted["predict"], U_test
    )
    metrics = rollout_l2_metrics(prediction, U_test)
    forecast = forecast_only_full_l2(prediction, U_test)
    mass = mass_per_timestep_np(prediction)
    mass_drift = np.mean(np.abs(mass - mass[:, :1]))

    result = {
        "n_train": int(n_train),
        "model": f"AWFNO-H{HISTORY_LENGTH}-K{WINDOW_K}-W{WIDTH}-L{N_LAYERS}",
        "parameters": int(count_params(fitted["model"])),
        "stored_real_scalars": int(count_real_scalars(fitted["model"])),
        "best_epoch": int(fitted["best_epoch"]),
        "validation_window_l2": float(fitted["best_val"]),
        "full_mean": metrics["full_l2_mean"],
        "full_std": metrics["full_l2_std"],
        "full_median": metrics["full_l2_median"],
        "forecast_full_mean": forecast["forecast_full_mean"],
        "forecast_full_std": forecast["forecast_full_std"],
        "forecast_full_median": forecast["forecast_full_median"],
        "final_mean": metrics["final_l2_mean"],
        "final_std": metrics["final_l2_std"],
        "final_median": metrics["final_l2_median"],
        "mean_mass_drift": float(mass_drift),
        "train_time_sec": float(fitted["train_time_sec"]),
        "inference_time_median_sec": float(inference_median),
        "inference_time_all_sec": inference_all,
        "checkpoint": fitted["checkpoint"],
    }
    all_results.append(result)

    np.savez_compressed(
        RESULTS_DIR / f"N{n_train}_metrics.npz",
        mean_t=metrics["mean_t"],
        median_t=metrics["median_t"],
        std_t=metrics["std_t"],
        all_full=metrics["all_full_l2"],
        all_final=metrics["all_final_l2"],
        inference_times=np.asarray(inference_all),
    )
    if SAVE_ROLLOUT_ARRAYS:
        np.save(
            RESULTS_DIR / f"N{n_train}_rollout.npy",
            prediction.astype(np.float32),
        )
    plot_history(fitted["history"], n_train)
    plot_l2_time(metrics, n_train)

    print("\nTest result")
    print("-----------")
    print(f"model             : {result['model']}")
    print(f"N_train           : {n_train}")
    print(f"parameters        : {result['parameters']:,}")
    print(f"stored real       : {result['stored_real_scalars']:,}")
    print(f"full mean/std     : {result['full_mean']:.6e} / {result['full_std']:.6e}")
    print(f"forecast-only mean: {result['forecast_full_mean']:.6e}")
    print(f"full median       : {result['full_median']:.6e}")
    print(f"final mean/std    : {result['final_mean']:.6e} / {result['final_std']:.6e}")
    print(f"final median      : {result['final_median']:.6e}")
    print(f"mean mass drift   : {result['mean_mass_drift']:.6e}")
    print(f"training time     : {result['train_time_sec']:.2f} s")
    print(f"inference median  : {result['inference_time_median_sec']:.4f} s")

    del train_history, train_future, fitted, prediction, metrics, forecast
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

csv_path = RESULTS_DIR / "awfno_h10_k10_r2_sample_size_summary.csv"
json_path = RESULTS_DIR / "awfno_h10_k10_r2_sample_size_summary.json"
try:
    import pandas as pd
    pd.DataFrame(all_results).to_csv(csv_path, index=False)
    display(pd.DataFrame(all_results))
except Exception:
    import csv
    with csv_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(all_results[0].keys()))
        writer.writeheader()
        writer.writerows(all_results)

with json_path.open("w") as handle:
    json.dump(all_results, handle, indent=2, default=json_safe)

print("\n" + "=" * 145)
print("CAHN--HILLIARD 2D AWFNO-H10-K10-R2: ABSOLUTE PHYSICAL L2")
print("=" * 145)
for row in all_results:
    print(
        f"N={row['n_train']:4d} | "
        f"full mean {row['full_mean']:.6e} | std {row['full_std']:.6e} | "
        f"median {row['full_median']:.6e} | "
        f"final mean {row['final_mean']:.6e} | "
        f"final median {row['final_median']:.6e} | "
        f"params {row['parameters']:,} | train {row['train_time_sec']:.2f}s | "
        f"inference {row['inference_time_median_sec']:.4f}s"
    )
print("=" * 145)
print("Saved CSV :", csv_path)
print("Saved JSON:", json_path)



Loading split cache
-------------------
/content/drive/MyDrive/anus/cahn_hilliard_awfno_h10_k10_r2_v1/AWFNO_H10_K10_W32_L4_mx16_my16_R2_E200_sizes25-50-100-200-400_mean1/data/split_CH2D_104d52baa9.npz

Geometry
--------
N, Nx, Ny, Nt: 400 32 32 101
x range      : -1.0 1.0
y range      : -1.0 1.0
t range      : 0.0 1.0
DX           : 0.06451612903225806
DY           : 0.06451612903225806
DT           : 0.01

Exact overlap check train vs validation
------------------------------------------
duplicates: 0

Exact overlap check train vs test
------------------------------------------
duplicates: 0

Exact overlap check validation vs test
------------------------------------------
duplicates: 0

Temporal protocol
-----------------
exact input states : 10
predicted per call : 10
training starts    : [0, 10, 20, 30, 40, 50, 60, 70, 80]
test forecast      : t[10:] recursively; t[:10] copied exactly

Shape smoke test passed: (2, 32, 32, 10)
Trainable parameters     : 2,308,266
Stored real scalar

,n_train,model,parameters,stored_real_scalars,best_epoch,validation_window_l2,full_mean,full_std,full_median,forecast_full_mean,forecast_full_std,forecast_full_median,final_mean,final_std,final_median,mean_mass_drift,train_time_sec,inference_time_median_sec,inference_time_all_sec,checkpoint
0,25,AWFNO-H10-K10-W32-L4,2308266,4405418,195,0.015298,0.480125,0.115198,0.483569,0.480125,0.115198,0.483569,0.819142,0.227787,0.827410,0.000109,54.198299,0.047945,"[0.04896569299990006, 0.04794535300004554, 0.0...",/content/drive/MyDrive/anus/cahn_hilliard_awfn...
1,50,AWFNO-H10-K10-W32-L4,2308266,4405418,189,0.011434,0.372003,0.102364,0.363249,0.372003,0.102364,0.363249,0.659150,0.197174,0.663316,0.000109,37.003579,0.048131,"[0.04934985200009123, 0.04813119300001745, 0.0...",/content/drive/MyDrive/anus/cahn_hilliard_awfn...
2,100,AWFNO-H10-K10-W32-L4,2308266,4405418,193,0.007867,0.278285,0.085065,0.261029,0.278285,0.085065,0.261029,0.525513,0.179586,0.519504,0.000109,49.294179,0.047865,"[0.04907840199996372, 0.047864672999935465, 0....",/content/drive/MyDrive/anus/cahn_hilliard_awfn...
3,200,AWFNO-H10-K10-W32-L4,2308266,4405418,198,0.004917,0.194463,0.080631,0.172816,0.194463,0.080631,0.172816,0.359167,0.172183,0.296668,0.000109,87.201727,0.048536,"[0.048759431999997105, 0.04853608299993084, 0....",/content/drive/MyDrive/anus/cahn_hilliard_awfn...
4,400,AWFNO-H10-K10-W32-L4,2308266,4405418,200,0.003047,0.128735,0.074536,0.100776,0.128735,0.074536,0.100776,0.240110,0.171123,0.157161,0.000109,167.437653,0.046117,"[0.047265892999917014, 0.04611691399998108, 0....",/content/drive/MyDrive/anus/cahn_hilliard_awfn...



CAHN--HILLIARD 2D AWFNO-H10-K10-R2: ABSOLUTE PHYSICAL L2
N=  25 | full mean 4.801249e-01 | std 1.151977e-01 | median 4.835691e-01 | final mean 8.191418e-01 | final median 8.274103e-01 | params 2,308,266 | train 54.20s | inference 0.0479s
N=  50 | full mean 3.720026e-01 | std 1.023638e-01 | median 3.632487e-01 | final mean 6.591504e-01 | final median 6.633161e-01 | params 2,308,266 | train 37.00s | inference 0.0481s
N= 100 | full mean 2.782849e-01 | std 8.506453e-02 | median 2.610293e-01 | final mean 5.255134e-01 | final median 5.195044e-01 | params 2,308,266 | train 49.29s | inference 0.0479s
N= 200 | full mean 1.944633e-01 | std 8.063097e-02 | median 1.728157e-01 | final mean 3.591665e-01 | final median 2.966681e-01 | params 2,308,266 | train 87.20s | inference 0.0485s
N= 400 | full mean 1.287348e-01 | std 7.453635e-02 | median 1.007758e-01 | final mean 2.401100e-01 | final median 1.571607e-01 | params 2,308,266 | train 167.44s | inference 0.0461s
Saved CSV : /content/drive/MyDrive/a

In [17]:
from google.colab import runtime
runtime.unassign()